# Exercise 4: Handling missing values

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
IRIS_SOURCE = DATA_DIR / "iris.csv"
if not IRIS_SOURCE.exists():
    IRIS_SOURCE = "https://raw.githubusercontent.com/01-edu/public/master/subjects/ai/pandas/data/iris.csv"

df = pd.read_csv(IRIS_SOURCE)
df = df.drop(columns="flower")
df = df.apply(pd.to_numeric, errors="coerce")
df.describe()

,sepal_length,sepal_width,petal_length,petal_width
count,146.000000,141.000000,120.000000,147.000000
mean,56.907534,52.625532,15.529167,12.026531
std,572.222221,417.127170,127.459631,131.873447
min,-4.400000,-3.600000,-4.800000,-2.500000
25%,5.100000,2.800000,2.725000,0.300000
50%,5.750000,3.000000,4.500000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,6900.000000,3809.000000,1400.000000,1600.000000


In [2]:
filled_by_strategy = df.fillna(
    {
        "sepal_length": df["sepal_length"].mean(),
        "sepal_width": df["sepal_width"].median(),
        "petal_length": 0,
        "petal_width": 0,
    }
)
filled_by_strategy.isna().sum()

sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
dtype: int64

In [3]:
filled_by_median = df.apply(lambda column: column.fillna(column.median()))
filled_by_median.isna().sum()

sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
dtype: int64

In [4]:
abnormal_rows = df[(df.lt(0) | df.gt(100)).any(axis=1)]
abnormal_rows

,sepal_length,sepal_width,petal_length,petal_width
4,5.0,-3.6,-1.4,0.2
8,-4.4,2.9,1400.0,0.2
25,5.0,-3.0,NaN,0.2
46,5.1,3809.0,1.6,0.2
51,6.4,3200.0,4.5,1.5
56,6.3,3.3,4.7,1600.0
126,6.2,2.8,-4.8,1.8
139,6900.0,3.1,5.4,2.1
142,580.0,2.7,5.1,NaN
144,6.7,3.3,5.7,-2.5


In [5]:
df.loc[122]

sepal_length   NaN
sepal_width    NaN
petal_length   NaN
petal_width    NaN
Name: 122, dtype: float64

Why `mean` and `0` can be bad here:

- the numeric summary is distorted by obvious outliers such as `6900`, `580`, `3809` and `1400`
- using the mean on a column with those values injects unrealistic flower sizes
- filling lengths and widths with `0` suggests a flower can physically have zero size, which does not make sense for this dataset
- row `122` is a fully missing row, and the abnormal rows show why checking the data before filling values matters